In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
paper_4.2.4.4_build_all.py
--------------------------
4.2.4.4節（OH系とFP系のクラスタ対応関係）の図表を一括生成する総合スクリプト。

出力構成（章番号入り・本文/付録を明示）:
  ROOT/paper_4.2.4.4_OHFP/
    ├─ main_text/
    │    ├─ FigSet_4.2.4.4_cumeig_gap/
    │    │    └─ Fig_4.2.4.4_OHFP-contrast_ARI-bars.pdf
    │    ├─ FigSet_4.2.4.4_cumeig_db/
    │    │    └─ Fig_4.2.4.4_OHFP-contrast_ARI-bars.pdf
    │    └─ Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv
    ├─ appendix/
    │    ├─ Appendix_Fig_allK_OHFP-contrast_ARI-bars.(pdf|png)
    │    └─ Appendix_Table_allK_OHFP-contrast_summary.csv
    ├─ reverse/
    │    ├─ main_text/
    │    │    ├─ FigSet_4.2.4.4R_cumeig_gap/  ← 逆方向の本文図（散布/コヒージョンなど）
    │    │    └─ FigSet_4.2.4.4R_cumeig_db/
    │    ├─ appendix/
    │    │    └─ FigSet_4.2.4.4R_{mode}_{index}/…
    │    └─ analysis_csv/
    │         ├─ Table_4.2.4.4R_fragment-to-OHmajor_{mode}_{index}.csv
    │         ├─ Table_4.2.4.4R_fragment-pair_cohesion_{mode}_{index}.csv
    │         └─ Table_4.2.4.4R_FPcluster_cohesion_{mode}_{index}.csv
    └─ bidirectional_summary/   ←（任意）双方向まとめ
         ├─ Table_4.2.4.4_bidirectional_summary.csv
         └─ Fig_4.2.4.4_bidirectional_comparison.pdf

前提:
  - figs_OH/figs_FP の ClusterAssign_*（OH/FP両方・top3/cumeig × index 一式）
  - STEP3.2_signlessCorr_MDS_YYYYmmdd/{OH,FP}/cluster_labels_*.csv（signless）
  - features_OH_onlyMat.csv / features_FP_onlyMat.csv（材料列のみ; 行=sample_id, 列=0/1）

NOTE:
  - 既存の個別スクリプト（4.2.4.3/4.2.4.4R）で使った関数を再掲・統合
  - 実行時は ROOT/SIGNLESS_BASE を環境に合わせて微調整してください
"""

from pathlib import Path
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from matplotlib.colors import to_rgb
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, adjusted_mutual_info_score
from itertools import combinations
from scipy.spatial.distance import cosine

# ========= ユーザー設定 =========
ROOT = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")
FIGS_OH = ROOT / "figs_OH"
FIGS_FP = ROOT / "figs_FP"

# signless 側
SIGNLESS_BASE = ROOT / "STEP3.2_signlessCorr_MDS_20250904"  # 例
SIGNLESS_DIR = {"OH": SIGNLESS_BASE / "OH", "FP": SIGNLESS_BASE / "FP"}

# features（材料列のみ）
FEATURES_OH = ROOT / "features_OH_onlyMat.csv"
FEATURES_FP = ROOT / "features_FP_onlyMat.csv"

# 出力ルート
OUT_OHFP_ROOT     = ROOT / "paper_4.2.4.4_OHFP"
OUT_MAIN          = OUT_OHFP_ROOT / "main_text"
OUT_APPENDIX      = OUT_OHFP_ROOT / "appendix"
OUT_REVERSE_ROOT  = OUT_OHFP_ROOT / "reverse"
OUT_R_MAIN        = OUT_REVERSE_ROOT / "main_text"
OUT_R_APPENDIX    = OUT_REVERSE_ROOT / "appendix"
OUT_R_CSV         = OUT_REVERSE_ROOT / "analysis_csv"
OUT_BIDIR         = OUT_OHFP_ROOT / "bidirectional_summary"
for d in [OUT_MAIN, OUT_APPENDIX, OUT_R_MAIN, OUT_R_APPENDIX, OUT_R_CSV, OUT_BIDIR]:
    d.mkdir(parents=True, exist_ok=True)

# 対象の mode/index
DIMS    = ["top3","cumeig"]
INDICES = ["silhouette","dunn","gap","ch","db","ptbiserial"]
MAIN_PAIRS = {("cumeig","gap"), ("cumeig","db")}  # 本文優先

K_MAX_KEEP = 30
DPI = 300

# ========= フォント =========
def set_font():
    cand = ["Noto Sans CJK JP","Noto Sans JP","Source Han Sans JP","IPAexGothic","Yu Gothic","Meiryo","Hiragino Sans","Meiryo UI"]
    have = set()
    for p in fm.findSystemFonts(fontext="ttf"):
        try: have.add(fm.FontProperties(fname=p).get_name())
        except: pass
    for w in cand:
        if any(w.lower() in h.lower() for h in have):
            matplotlib.rcParams["font.family"] = w
            break
    matplotlib.rcParams["axes.unicode_minus"] = False
    matplotlib.rcParams.update({"font.size":10, "axes.titlesize":12, "axes.labelsize":10, "legend.fontsize":9})

# ========= 汎用IO =========
def read_csv_quiet(path: Path) -> pd.DataFrame | None:
    if not path or not path.exists(): return None
    for enc in ("utf-8","utf-8-sig","cp932"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            continue
    return None

# ========= 4.2.4.4（OH→FP）: new vs signless の一致度をデータセットごとに集計・図化 =========
def collect_assign_labels(figs_dir: Path, ds: str) -> pd.DataFrame | None:
    pat = re.compile(rf"^ClusterAssign_(top3|cumeig)_(silhouette|dunn|gap|ch|db|ptbiserial)_{ds}_.*\.csv$")
    files = [p for p in figs_dir.glob("ClusterAssign_*.csv") if pat.match(p.name)]
    parts=[]
    for f in files:
        m = re.match(r"ClusterAssign_(top3|cumeig)_([A-Za-z]+)_", f.name)
        if not m: continue
        mode, index = m.groups()
        df = read_csv_quiet(f)
        if df is None: continue
        if {"Variable","Cluster"}.issubset(df.columns):
            ids = df["Variable"].astype(str).values
            cl  = df["Cluster"].values
        elif df.shape[1]==1:
            ids = df.index.astype(str).values
            cl  = df.iloc[:,0].values
        else:
            continue
        codes = pd.Categorical(cl).codes
        codes = pd.Series(codes).replace({-1: np.nan}).values
        parts.append(pd.DataFrame({"id": ids, f"{mode}_{index}": codes}))
    if not parts: return None
    out = parts[0]
    for p in parts[1:]:
        out = out.merge(p, on="id", how="outer")
    return out.drop_duplicates(subset=["id"])

def load_signless_labels(signless_dir: Path) -> pd.DataFrame | None:
    cands = sorted(signless_dir.glob("cluster_labels_*.csv"), key=os.path.getmtime, reverse=True)
    if not cands: return None
    df = read_csv_quiet(cands[0])
    if df is None or "id" not in df.columns: return None
    keep = [c for c in df.columns if str(c).startswith("linear_")]
    if not keep: return None
    out = df[["id"]+keep].copy()
    out.columns = ["id"] + [c.replace("linear_","",1) for c in keep]
    return out

def pairwise_scores(dfA: pd.DataFrame|None, dfB: pd.DataFrame|None) -> pd.DataFrame:
    cols = ["condition","n","kA","kB","ARI","NMI","AMI"]
    if dfA is None or dfB is None: return pd.DataFrame(columns=cols)
    colsA = [c for c in dfA.columns if c!="id"]
    colsB = [c for c in dfB.columns if c!="id"]
    common = sorted(set(colsA).intersection(colsB))
    if not common: return pd.DataFrame(columns=cols)
    merged = dfA.merge(dfB, on="id", suffixes=(".A",".B"))
    rows=[]
    for cn in common:
        a = merged[f"{cn}.A"].values
        b = merged[f"{cn}.B"].values
        mask = ~pd.isna(a) & ~pd.isna(b)
        if mask.sum() < 2: continue
        a = a[mask].astype(int); b = b[mask].astype(int)
        rows.append([
            cn, int(mask.sum()),
            int(len(np.unique(a))), int(len(np.unique(b))),
            float(adjusted_rand_score(a,b)),
            float(normalized_mutual_info_score(a,b)),
            float(adjusted_mutual_info_score(a,b)),
        ])
    return pd.DataFrame(rows, columns=cols)

def lighten(color, factor=0.5):
    r,g,b = matplotlib.colors.to_rgb(color)
    return (1 - factor) + factor*r, (1 - factor) + factor*g, (1 - factor) + factor*b

def save_bar_contrast_with_k(df_long: pd.DataFrame, T_meta: pd.DataFrame, out_dir: Path, title: str, fname_core: str, fade_outside: bool):
    set_font()
    INDEX_LIST = ["silhouette","dunn","gap","ch","db","ptbiserial"]
    DATASETS = ["OH","FP"]

    df = df_long.copy()
    df["mode"]  = pd.Categorical(df["mode"], categories=["top3","cumeig"], ordered=True)
    df["index"] = pd.Categorical(df["index"], categories=INDEX_LIST, ordered=True)
    df = df.sort_values(["index","mode","Dataset"])

    # x slots
    x_labels = [f"{m}\n{ix}" for m in ["top3","cumeig"] for ix in INDEX_LIST]
    grid=[]
    for m in ["top3","cumeig"]:
        for ix in INDEX_LIST:
            sl = df[(df["mode"]==m) & (df["index"]==ix)]
            grid.append(sl.set_index("Dataset")["ARI"].reindex(DATASETS))
    X = pd.concat(grid, axis=1).T
    X.index = x_labels

    # META lookup
    M = T_meta.copy()
    M["row_key"] = M["mode"] + "\n" + M["index"]
    M["is_target"] = (M["kA"]<=K_MAX_KEEP) & (M["kB"]<=K_MAX_KEEP)
    META = (M.set_index(["row_key","Dataset"])[["kA","kB","is_target"]]
              .reindex(pd.MultiIndex.from_product([x_labels, DATASETS], names=["row_key","Dataset"])))

    color_OH = "#1f77b4"; color_FP="#ff7f0e"
    fade_OH  = lighten(color_OH, 0.7); fade_FP = lighten(color_FP, 0.7)

    fig, ax = plt.subplots(figsize=(12,6), dpi=DPI)
    width = 0.38; x = np.arange(X.shape[0])

    for i, ds in enumerate(DATASETS):
        base_color = color_OH if ds=="OH" else color_FP
        base_fade  = fade_OH  if ds=="OH" else fade_FP
        xs = x + (i-0.5)*width
        vals = X[ds].values
        for xi, val, row_key in zip(xs, vals, X.index):
            if (row_key, ds) in META.index:
                kA = META.loc[(row_key, ds), "kA"]
                kB = META.loc[(row_key, ds), "kB"]
                is_t = bool(META.loc[(row_key, ds), "is_target"])
            else:
                kA = np.nan; kB = np.nan; is_t=False
            fc = base_color if (not fade_outside or is_t) else base_fade
            alpha = 0.95 if (not fade_outside or is_t) else 0.5
            ax.bar(xi, val, width, color=fc, alpha=alpha, edgecolor="none")
            if not np.isnan(val):
                label = f"{val:.2f}\n({int(kA) if not np.isnan(kA) else '-'}" \
                        f"/{int(kB) if not np.isnan(kB) else '-'})"
                ax.text(xi, val + 0.015, label, ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x); ax.set_xticklabels(X.index, rotation=35, ha="right")
    ax.set_ylim(0,1.05); ax.set_ylabel("ARI")
    ax.set_title(title)
    from matplotlib.patches import Patch
    legend_elems = [
        Patch(facecolor=color_OH, edgecolor="none", alpha=0.95, label="OH (k≤30)"),
        Patch(facecolor=color_FP, edgecolor="none", alpha=0.95, label="FP (k≤30)")
    ]
    if fade_outside:
        legend_elems += [
            Patch(facecolor=fade_OH, edgecolor="none", alpha=0.50, label="OH (k>30)"),
            Patch(facecolor=fade_FP, edgecolor="none", alpha=0.50, label="FP (k>30)")
        ]
    ax.legend(handles=legend_elems, loc="upper left", bbox_to_anchor=(1.02, 1.0))
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
    fig.tight_layout()

    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / f"{fname_core}.png").with_suffix(".png")
    fig.savefig(out_dir / f"{fname_core}.png", bbox_inches="tight")
    fig.savefig(out_dir / f"{fname_core}.pdf", bbox_inches="tight")
    plt.close(fig)

def build_ohfp_contrast():
    # OH→FP（new vs signless 一致度）をデータセット別に棒グラフ化
    all_tbl_main=[]; all_for_contrast_main=[]; all_tbl_allK=[]; all_for_contrast_allK=[]
    for ds, figs in [("OH",FIGS_OH), ("FP",FIGS_FP)]:
        A = collect_assign_labels(figs, ds)
        S = load_signless_labels(SIGNLESS_DIR[ds])
        T = pairwise_scores(A, S)
        if T.empty: 
            print(f"[WARN] {ds}: no common conditions"); 
            continue
        T["mode"]    = T["condition"].str.replace(r"_(.*)$", "", regex=True)
        T["index"]   = T["condition"].str.replace(r"^(.*)_", "", regex=True)
        T["Dataset"] = ds
        T["k_max"]   = T[["kA","kB"]].max(axis=1)

        all_tbl_allK.append(T[["condition","Dataset","n","kA","kB","ARI","NMI","AMI","mode","index","k_max"]])
        all_for_contrast_allK.append(T[["Dataset","mode","index","ARI","kA","kB","k_max"]])

        T30 = T[(T["k_max"].isna()) | (T["k_max"]<=K_MAX_KEEP)].copy()
        if len(T30):
            all_tbl_main.append(T30[["condition","Dataset","n","kA","kB","ARI","NMI","AMI","mode","index","k_max"]])
            all_for_contrast_main.append(T30[["Dataset","mode","index","ARI","kA","kB","k_max"]])

    # 本文（k≤30）
    if all_tbl_main:
        T_all = pd.concat(all_tbl_main, ignore_index=True)
        out_csv = OUT_MAIN / "Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv"
        T_all.sort_values(["Dataset","mode","index"]).to_csv(out_csv, index=False, encoding="utf-8-sig")
        print(f"[SAVE] {out_csv}")

        contrast_src = pd.concat(all_for_contrast_main, ignore_index=True)
        for mode,index in MAIN_PAIRS:
            sub = contrast_src[(contrast_src["mode"]==mode)&(contrast_src["index"]==index)]
            out_dir = OUT_MAIN / f"FigSet_4.2.4.4_{mode}_{index}"
            save_bar_contrast_with_k(
                df_long=sub,
                T_meta=T_all[T_all["mode"]==mode][T_all["index"]==index][["Dataset","mode","index","kA","kB","k_max"]],
                out_dir=out_dir,
                title=f"Fig. 4.2.4.4: OH vs FP (ARI of new vs signless, {mode}×{index}, k≤30)",
                fname_core="Fig_4.2.4.4_OHFP-contrast_ARI-bars",
                fade_outside=False
            )
    else:
        print("[MAIN] No k≤30 rows for OH→FP contrast.")

    # 付録（all k）
    if all_tbl_allK:
        T_allK = pd.concat(all_tbl_allK, ignore_index=True)
        out_csv_apx = OUT_APPENDIX / "Appendix_Table_allK_OHFP-contrast_summary.csv"
        T_allK.sort_values(["Dataset","mode","index"]).to_csv(out_csv_apx, index=False, encoding="utf-8-sig")
        print(f"[SAVE] {out_csv_apx}")

        # 付録向け：全条件まとめ棒（淡色：k>30）
        contrast_allK = pd.concat(all_for_contrast_allK, ignore_index=True)
        save_bar_contrast_with_k(
            df_long=contrast_allK[["Dataset","mode","index","ARI"]],
            T_meta=T_allK[["Dataset","mode","index","kA","kB","k_max"]],
            out_dir=OUT_APPENDIX,
            title="Appendix: OH vs FP (ARI of new vs signless, all k)",
            fname_core="Appendix_Fig_allK_OHFP-contrast_ARI-bars",
            fade_outside=True
        )

# ========= 4.2.4.4R（FP→OH 逆方向） =========
FN_RE = re.compile(
    r"^ClusterAssign_(?P<mode>top3|cumeig)_(?P<index>silhouette|dunn|gap|ch|db|ptbiserial)_(?P<ds>OH|FP)_(?P<date>\d{8})_(?P<hm>\d{4})\.csv$"
)
def latest_by_mode_index(fig_dir: Path, ds: str):
    latest = {}
    for p in fig_dir.glob("ClusterAssign_*.csv"):
        m = FN_RE.match(p.name)
        if not m or m["ds"] != ds: 
            continue
        key = (m["mode"], m["index"])
        ts  = f"{m['date']}_{m['hm']}"
        if key not in latest or ts > latest[key][0]:
            latest[key] = (ts, p)
    return {k: v[1] for k,v in latest.items()}

def read_var_cluster(path: Path) -> pd.Series:
    df = read_csv_quiet(path)
    cols = {c.lower(): c for c in df.columns}
    vcol = cols.get("variable", df.columns[0])
    ccol = cols.get("cluster",  df.columns[-1])
    s = pd.Series(df[ccol].values, index=df[vcol].astype(str).values, name="cluster")
    try: s = s.astype(int)
    except: pass
    return s

def load_features_bin(path: Path) -> pd.DataFrame:
    df = read_csv_quiet(path)
    if df is None: raise FileNotFoundError(path)
    if df.columns[0].lower()!="sample_id":
        df = df.rename(columns={df.columns[0]:"sample_id"})
    X = df.set_index("sample_id").apply(pd.to_numeric, errors="coerce").fillna(0.0)
    return (X!=0).astype(int)

def entropy_from_dist(dist: dict[int, float]) -> float:
    if not dist: return np.nan
    p = np.array(list(dist.values()), dtype=float)
    p = p / (p.sum() + 1e-12)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def js_divergence(p, q):
    p = np.asarray(p, dtype=float); p = p/p.sum() if p.sum()>0 else p
    q = np.asarray(q, dtype=float); q = q/q.sum() if q.sum()>0 else q
    m = 0.5*(p+q)
    def _kl(a,b):
        a = np.clip(a,1e-12,1); b = np.clip(b,1e-12,1)
        return float(np.sum(a*(np.log2(a)-np.log2(b))))
    return 0.5*_kl(p,m) + 0.5*_kl(q,m)

def fragment_to_oh_distribution(fragment, Xfp, Xoh, oh_var2clu, threshold=0.20):
    idx = Xfp.index[Xfp[fragment] == 1]
    if len(idx) == 0: return None, None, 0, {}
    prev = Xoh.loc[idx].mean(axis=0)       # P(material=1 | fragment=1)
    sel = prev[prev >= threshold]
    if sel.empty: return None, None, len(idx), {}
    df = pd.DataFrame({"mat": sel.index, "w": sel.values})
    df["oh_cluster"] = df["mat"].map(oh_var2clu).astype("Int64")
    df = df.dropna(subset=["oh_cluster"])
    if df.empty: return None, None, len(idx), {}
    grp = df.groupby("oh_cluster", as_index=False)["w"].sum()
    grp["p"] = grp["w"]/grp["w"].sum()
    dist = dict(zip(grp["oh_cluster"].astype(int), grp["p"]))
    maj = int(grp.sort_values("p", ascending=False).iloc[0]["oh_cluster"])
    purity = float(grp["p"].max())
    return maj, purity, len(idx), dist

def plot_fragment_scatter(df_frag: pd.DataFrame, outdir: Path, mode: str, index: str, top_labels:int=15, main_text:bool=False):
    set_font()
    if df_frag.empty: return
    size = 30 + 3 * df_frag["n_samples_with_fragment"].fillna(0).astype(float)
    c    = df_frag["FP_cluster"].astype("category")
    fig, ax = plt.subplots(figsize=(9,6), dpi=DPI)
    sc = ax.scatter(df_frag["OH_entropy"], df_frag["OH_purity"], c=c.cat.codes, s=size, cmap="tab20", alpha=0.85, edgecolors="none")
    ax.set_xlabel("OH entropy (larger = more materials-spread)")
    ax.set_ylabel("OH purity (larger = more materials-specific)")
    title = f"Fig 4.2.4.4R: Fragments — Specificity vs Spread ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title)
    lab_df = df_frag.sort_values(["OH_purity","n_samples_with_fragment"], ascending=False).head(top_labels)
    for _, r in lab_df.iterrows():
        ax.text(r["OH_entropy"], r["OH_purity"]+0.015, str(r["fragment"]), ha="center", va="bottom", fontsize=8)
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
    outdir.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fname = f"Fig_4.2.4.4R_fragments_specificity-vs-spread_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight")
    fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)

def plot_pair_distributions(df_pair: pd.DataFrame, outdir: Path, mode: str, index: str, main_text:bool=False):
    if df_pair is None or df_pair.empty: return
    set_font()
    # major same
    fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
    rate = df_pair["OH_major_same"].value_counts(normalize=True).rename({True:"Same", False:"Different"})
    idx = ["Same","Different"]; vals = [rate.get("Same",0.0), rate.get("Different",0.0)]
    ax.bar(idx, vals, color=["#4E79A7","#E15759"], alpha=0.9)
    ax.set_ylim(0,1); ax.set_ylabel("Proportion")
    title = f"Fig 4.2.4.4R: Pair-level — OH major same? ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title)
    for xi, val in enumerate(vals):
        ax.text(xi, val+0.02, f"{val:.2f}", ha="center", va="bottom")
    fig.tight_layout()
    fname = f"Fig_4.2.4.4R_pair_major-same_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight")
    fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)
    # cosine / JS
    for col, xlabel in [("OH_cosine_similarity","Cosine similarity (1=similar)"),
                        ("OH_JS_divergence","JS divergence (0=similar)")]:
        s = df_pair[col].dropna()
        if not len(s): continue
        fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
        ax.hist(s, bins=20, alpha=0.9)
        title = f"Fig 4.2.4.4R: Pair-level — {col} ({mode} × {index})"
        if main_text: title += " [Main]"
        ax.set_title(title)
        ax.set_xlabel(xlabel); ax.set_ylabel("Count")
        fig.tight_layout()
        fname = f"Fig_4.2.4.4R_pair_{col}_hist_{mode}_{index}{'_MAIN' if main_text else ''}"
        fig.savefig(outdir / f"{fname}.png", bbox_inches="tight")
        fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
        plt.close(fig)

def plot_fpcluster_consistency(df_fpc: pd.DataFrame, outdir: Path, mode: str, index: str, main_text:bool=False):
    if df_fpc is None or df_fpc.empty: return
    set_font()
    df = df_fpc.sort_values("cohesive_ratio", ascending=False).copy()
    fig, ax = plt.subplots(figsize=(8,4), dpi=DPI)
    x = np.arange(len(df))
    ax.bar(x, df["cohesive_ratio"].values, color="#59A14F", alpha=0.9)
    ax.set_xticks(x); ax.set_xticklabels([f"FP {int(c)}" for c in df["FP_cluster"]], rotation=35, ha="right")
    ax.set_ylim(0,1.05)
    ax.set_ylabel("Cohesive ratio (pair-level in OH space)")
    title = f"Fig 4.2.4.4R: FP-cluster cohesion in OH space ({mode} × {index})"
    if main_text: title += " [Main]"
    ax.set_title(title)
    for xi, v in zip(x, df["cohesive_ratio"].values):
        ax.text(xi, v+0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
    fig.tight_layout()
    fname = f"Fig_4.2.4.4R_FPcluster_cohesion_{mode}_{index}{'_MAIN' if main_text else ''}"
    fig.savefig(outdir / f"{fname}.png", bbox_inches="tight")
    fig.savefig(outdir / f"{fname}.pdf", bbox_inches="tight")
    plt.close(fig)

def build_reverse_direction():
    set_font()
    Xoh0 = load_features_bin(FEATURES_OH)
    Xfp0 = load_features_bin(FEATURES_FP)
    common = Xoh0.index.intersection(Xfp0.index)
    if len(common)==0: 
        raise RuntimeError("No common samples between features_OH_onlyMat and features_FP_onlyMat.")
    Xoh0 = Xoh0.loc[common]; Xfp0 = Xfp0.loc[common]
    oh_latest = latest_by_mode_index(FIGS_OH, "OH")
    fp_latest = latest_by_mode_index(FIGS_FP, "FP")
    keys = [(m,i) for m in DIMS for i in INDICES if (m,i) in oh_latest and (m,i) in fp_latest]
    if not keys: 
        raise RuntimeError("No common (mode,index) between figs_OH and figs_FP.")

    summary_rows=[]
    for (mode,index) in keys:
        oh_assign = read_var_cluster(oh_latest[(mode,index)])
        fp_assign = read_var_cluster(fp_latest[(mode,index)])

        # フラグメント → OH 主分布
        fragments = [f for f in fp_assign.index if f in Xfp0.columns]
        rows_frag=[]
        for frag in fragments:
            n_frag = int(Xfp0[frag].sum())
            if n_frag<=0: 
                continue
            maj, purity, n_s, dist = fragment_to_oh_distribution(frag, Xfp0, Xoh0, oh_assign, threshold=0.20)
            fp_c = fp_assign.get(frag, np.nan)
            H = entropy_from_dist(dist)
            k_eff = len(dist) if dist else 0
            rows_frag.append({
                "mode":mode,"index":index,"fragment":frag,
                "FP_cluster": int(fp_c) if pd.notna(fp_c) else np.nan,
                "n_samples_with_fragment": n_s,
                "OH_major_cluster": maj,
                "OH_purity": purity,
                "OH_entropy": H,
                "OH_k_effective": k_eff,
                "OH_dist": ";".join(f"{k}:{v:.3f}" for k,v in sorted(dist.items())) if dist else ""
            })
        df_frag = pd.DataFrame(rows_frag)
        out_csv1 = OUT_R_CSV / f"Table_4.2.4.4R_fragment-to-OHmajor_{mode}_{index}.csv"
        if len(df_frag): 
            df_frag.to_csv(out_csv1, index=False)

        # 同一FPクラスタ内のフラグメントペアの材料側一貫性
        rows_pair=[]
        if len(df_frag):
            def parse_dist(s):
                if isinstance(s,str) and s:
                    d={}
                    for t in s.split(";"):
                        k,v = t.split(":"); d[int(k)]=float(v)
                    return d
                return {}
            dists = {r.fragment: parse_dist(r.OH_dist) for r in df_frag.itertuples()}
            all_oh_clusters = sorted({k for d in dists.values() for k in d.keys()})
            def vec_of(dist):
                return np.array([dist.get(k, 0.0) for k in all_oh_clusters], dtype=float)

            for k, grp in df_frag.dropna(subset=["FP_cluster"]).groupby("FP_cluster"):
                frags = grp["fragment"].tolist()
                for a,b in combinations(frags,2):
                    da, db = dists.get(a,{}), dists.get(b,{})
                    va, vb = vec_of(da), vec_of(db)
                    if va.sum()==0 and vb.sum()==0:
                        cos_sim=np.nan; jsd=np.nan
                    else:
                        cos_sim = 1.0 - (cosine(va, vb) if (va.sum()>0 and vb.sum()>0) else 1.0)
                        jsd = js_divergence(va, vb) if (va.sum()>0 and vb.sum()>0) else np.nan
                    maj_same = (df_frag.loc[df_frag["fragment"]==a,"OH_major_cluster"].values[0] ==
                                df_frag.loc[df_frag["fragment"]==b,"OH_major_cluster"].values[0])
                    purity_min = float(min(
                        df_frag.loc[df_frag["fragment"]==a,"OH_purity"].values[0] if len(df_frag.loc[df_frag["fragment"]==a,"OH_purity"]) else 0.0,
                        df_frag.loc[df_frag["fragment"]==b,"OH_purity"].values[0] if len(df_frag.loc[df_frag["fragment"]==b,"OH_purity"]) else 0.0
                    ))
                    rows_pair.append({
                        "mode":mode,"index":index,
                        "FP_cluster":int(k),
                        "fragment_A":a,"fragment_B":b,
                        "OH_major_same":bool(maj_same),
                        "OH_purity_min": purity_min,
                        "OH_cosine_similarity": cos_sim,
                        "OH_JS_divergence": jsd
                    })
        df_pair = pd.DataFrame(rows_pair)
        out_csv2 = OUT_R_CSV / f"Table_4.2.4.4R_fragment-pair_cohesion_{mode}_{index}.csv"
        if len(df_pair): df_pair.to_csv(out_csv2, index=False)

        if len(df_pair):
            def label_cohesion(r, purity_thr=0.60, cos_thr=0.60, js_thr=0.50):
                return bool(
                    (r["OH_major_same"] and r["OH_purity_min"]>=purity_thr) or
                    (pd.notna(r["OH_cosine_similarity"]) and r["OH_cosine_similarity"]>=cos_thr) or
                    (pd.notna(r["OH_JS_divergence"]) and r["OH_JS_divergence"]<=js_thr)
                )
            df_pair["cohesive_flag"] = df_pair.apply(label_cohesion, axis=1)
            df_fpc = (df_pair.groupby(["mode","index","FP_cluster"], as_index=False)
                             .agg(n_pairs=("cohesive_flag","size"),
                                  n_cohesive=("cohesive_flag","sum")))
            df_fpc["cohesive_ratio"] = df_fpc["n_cohesive"]/df_fpc["n_pairs"].replace(0,np.nan)
        else:
            df_fpc = pd.DataFrame()
        out_csv3 = OUT_R_CSV / f"Table_4.2.4.4R_FPcluster_cohesion_{mode}_{index}.csv"
        if len(df_fpc): df_fpc.to_csv(out_csv3, index=False)

        # 図（本文/付録 へ振分）
        is_main = (mode,index) in MAIN_PAIRS
        out_dir = (OUT_R_MAIN if is_main else OUT_R_APPENDIX) / f"FigSet_4.2.4.4R_{mode}_{index}"
        out_dir.mkdir(parents=True, exist_ok=True)
        plot_fragment_scatter(df_frag, out_dir, mode, index, top_labels=15, main_text=is_main)
        plot_pair_distributions(df_pair, out_dir, mode, index, main_text=is_main)
        plot_fpcluster_consistency(df_fpc, out_dir, mode, index, main_text=is_main)

        # サマリー行
        summary_rows.append({
            "mode":mode,"index":index,
            "n_fragments": len(df_frag),
            "mean_OH_purity": df_frag["OH_purity"].dropna().mean() if "OH_purity" in df_frag else np.nan,
            "mean_OH_entropy": df_frag["OH_entropy"].dropna().mean() if "OH_entropy" in df_frag else np.nan,
            "pair_OH_major_same_rate": df_pair["OH_major_same"].mean() if len(df_pair) else np.nan,
            "pair_mean_cosine_similarity": df_pair["OH_cosine_similarity"].dropna().mean() if len(df_pair) else np.nan,
            "pair_mean_JS_divergence": df_pair["OH_JS_divergence"].dropna().mean() if len(df_pair) else np.nan,
            "FPcluster_median_cohesive_ratio": df_fpc["cohesive_ratio"].median() if len(df_fpc) else np.nan
        })

    if summary_rows:
        df_sum = pd.DataFrame(summary_rows).sort_values(["mode","index"])
        outsum = OUT_REVERSE_ROOT / "Table_4.2.4.4R_summary_all_conditions.csv"
        df_sum.to_csv(outsum, index=False)
        print(f"[SAVE] {outsum}")

# ========= 双方向まとめ（任意） =========
def build_bidirectional_summary():
    # OH→FP 側の本文表
    fwd = OUT_MAIN / "Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv"
    rev = OUT_REVERSE_ROOT / "Table_4.2.4.4R_summary_all_conditions.csv"
    if not fwd.exists() or not rev.exists():
        print("[INFO] Skip bidirectional summary (source tables not found).")
        return
    A = pd.read_csv(fwd)
    R = pd.read_csv(rev)

    # 前方向：各 (mode,index) で OH/FP の ARI を平均して簡易スコア化
    fwd_sc = (A.groupby(["mode","index"])["ARI"].mean().rename("forward_score")).reset_index()

    # 逆方向：purity↑・entropy↓・major_same↑・cosine↑・JS↓・cohesive_ratio↑ を 0-1正規化し平均
    def minmax(s):
        s2 = s.copy()
        if s2.isna().all(): return s2.fillna(0.5)
        s2 = s2.fillna(s2.mean()); mn, mx = s2.min(), s2.max()
        if np.isclose(mx, mn): return pd.Series(0.5, index=s2.index)
        return (s2 - mn) / (mx - mn)

    rev2 = R.copy()
    pos = [c for c in ["mean_OH_purity","pair_OH_major_same_rate","pair_mean_cosine_similarity","FPcluster_median_cohesive_ratio"] if c in rev2.columns]
    neg = [c for c in ["mean_OH_entropy","pair_mean_JS_divergence"] if c in rev2.columns]
    parts = []
    for c in pos: parts.append(minmax(rev2[c]))
    for c in neg: parts.append(1 - minmax(rev2[c]))
    if parts:
        rev2["reverse_score"] = pd.concat(parts, axis=1).mean(axis=1)
    else:
        rev2["reverse_score"] = np.nan
    rev_sc = rev2.groupby(["mode","index"])["reverse_score"].mean().reset_index()

    M = fwd_sc.merge(rev_sc, on=["mode","index"], how="outer").sort_values(["mode","index"])
    out_csv = OUT_BIDIR / "Table_4.2.4.4_bidirectional_summary.csv"
    M.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"[SAVE] {out_csv}")

    # 図：2軸バー（前方向 vs 逆方向）
    set_font()
    fig, ax = plt.subplots(figsize=(10,5), dpi=DPI)
    lbls = [f"{m}-{i}" for m,i in zip(M["mode"], M["index"])]
    x = np.arange(len(M))
    w = 0.38
    ax.bar(x-w/2, M["forward_score"].values, width=w, label="OH→FP (ARI mean)", alpha=0.9)
    ax.bar(x+w/2, M["reverse_score"].values, width=w, label="FP→OH (composite)", alpha=0.9)
    ax.set_xticks(x); ax.set_xticklabels(lbls, rotation=35, ha="right")
    ax.set_ylim(0,1.05); ax.set_ylabel("Score (0–1)")
    ax.set_title("Fig 4.2.4.4: Bidirectional alignment summary")
    ax.legend()
    ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
    fig.tight_layout()
    fig.savefig(OUT_BIDIR / "Fig_4.2.4.4_bidirectional_comparison.png", bbox_inches="tight")
    fig.savefig(OUT_BIDIR / "Fig_4.2.4.4_bidirectional_comparison.pdf", bbox_inches="tight")
    plt.close(fig)

# ========= MAIN =========
def main():
    set_font()
    print("=== [1/3] OH→FP contrast (new vs signless) ===")
    build_ohfp_contrast()
    print("=== [2/3] FP→OH reverse direction ===")
    build_reverse_direction()
    print("=== [3/3] Bidirectional summary (optional) ===")
    build_bidirectional_summary()
    print("\n✅ Done. Outputs under:", OUT_OHFP_ROOT)

if __name__ == "__main__":
    main()

=== [1/3] OH→FP contrast (new vs signless) ===
[WARN] OH: no common conditions
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/main_text/Table_4.2.4.4_OHFP-contrast_summary_k-le30.csv


/tmp/ipykernel_3936950/3169756695.py:292: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  T_meta=T_all[T_all["mode"]==mode][T_all["index"]==index][["Dataset","mode","index","kA","kB","k_max"]],
/tmp/ipykernel_3936950/3169756695.py:292: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  T_meta=T_all[T_all["mode"]==mode][T_all["index"]==index][["Dataset","mode","index","kA","kB","k_max"]],


[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/appendix/Appendix_Table_allK_OHFP-contrast_summary.csv
=== [2/3] FP→OH reverse direction ===
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/reverse/Table_4.2.4.4R_summary_all_conditions.csv
=== [3/3] Bidirectional summary (optional) ===
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP/bidirectional_summary/Table_4.2.4.4_bidirectional_summary.csv

✅ Done. Outputs under: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/paper_4.2.4.4_OHFP


In [3]:
# #!/usr/bin/env python3
# # -*- coding: utf-8 -*-

# """
# 4.2.4.4 OH–FP クラスタ対応（材料列のみ）図表一括生成
#  - Fig(b): FP purity vs FP entropy（色＝OHクラスタID、重複ラベル回避）
#  - Fig(c): OHクラスタの cohesive ratio（材料ペアのFPモード一致率）
#  - Fig(d): 同一OHクラスタ内の材料ペアの類似度分布（FP_major_same / cosine / JS）
#  - Table: 指標要約（cumeig×gap / db の主指標）

# 前提ファイル（存在必須）:
#   ROOT/features_OH_onlyMat.csv        # rows=Samples, cols=Materials (0/1)
#   ROOT/features_FP_onlyMat.csv        # rows=Samples, cols=Fragments (0/1)
#   ROOT/figs_OH/ClusterAssign_cumeig_db_OH_*.csv     # materials -> OH cluster（cumeig×db）
#   ROOT/figs_FP/ClusterAssign_cumeig_gap_FP_*.csv    # fragments -> FP cluster（cumeig×gap）

# 出力:
#   ROOT/OHFP_varmap/
#     ├─ figs_cumeig_gap/
#     │    ├─ Fig_material_scatter_entropy_vs_purity.pdf/png   [Fig 4.2.4.4(b)]
#     │    └─ Fig_pairwise_similarity_hist.pdf/png             [Fig 4.2.4.4(d)]
#     ├─ figs_cumeig_db/
#     │    └─ Fig_OHcluster_cohesive_ratio.pdf/png             [Fig 4.2.4.4(c)]
#     └─ Table_4.2.4.4_summary_all_conditions.csv              [本文テーブル]
# """

# from pathlib import Path
# import os, re, math
# import numpy as np
# import pandas as pd
# import matplotlib
# import matplotlib.pyplot as plt
# from matplotlib import font_manager as fm

# # ========== 基本設定 ==========
# ROOT = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No")
# FIGS_OH = ROOT / "figs_OH"
# FIGS_FP = ROOT / "figs_FP"

# OUT_BASE = ROOT / "OHFP_varmap"
# OUT_GAP  = OUT_BASE / "figs_cumeig_gap"
# OUT_DB   = OUT_BASE / "figs_cumeig_db"
# for d in (OUT_BASE, OUT_GAP, OUT_DB):
#     d.mkdir(parents=True, exist_ok=True)

# # 入力（材料列のみ）
# FEAT_OH = ROOT / "features_OH_onlyMat.csv"
# FEAT_FP = ROOT / "features_FP_onlyMat.csv"

# # ========== フォント ==========
# def _set_jp_font():
#     cand = ["Noto Sans CJK JP","Noto Sans JP","Source Han Sans JP","IPAexGothic",
#             "Yu Gothic","Meiryo","Hiragino Sans","Hiragino Kaku Gothic ProN"]
#     have = set()
#     for p in fm.findSystemFonts(fontext="ttf"):
#         try: have.add(fm.FontProperties(fname=p).get_name())
#         except: pass
#     for w in cand:
#         if any(w.lower() in h.lower() for h in have):
#             matplotlib.rcParams["font.family"] = w
#             break
#     matplotlib.rcParams["axes.unicode_minus"] = False
#     matplotlib.rcParams.update({"font.size":10,"axes.titlesize":12,"axes.labelsize":10,"legend.fontsize":9})

# # ========== IO ==========
# def read_csv_quiet(path: Path) -> pd.DataFrame | None:
#     if not path or not path.exists(): return None
#     for enc in ("utf-8","utf-8-sig","cp932"):
#         try:
#             return pd.read_csv(path, encoding=enc)
#         except Exception:
#             continue
#     return None

# def latest_match(dir_path: Path, pattern_regex: str) -> Path | None:
#     """正規表現で最新ファイル1つを返す"""
#     reg = re.compile(pattern_regex)
#     cands = [p for p in dir_path.glob("*.csv") if reg.match(p.name)]
#     if not cands: return None
#     return sorted(cands, key=os.path.getmtime, reverse=True)[0]

# # ========== 入力検証 ==========
# def ensure_inputs():
#     miss = []
#     if not FEAT_OH.exists(): miss.append(str(FEAT_OH))
#     if not FEAT_FP.exists(): miss.append(str(FEAT_FP))
#     f_oh = latest_match(FIGS_OH, r"^ClusterAssign_cumeig_db_OH_.+\.csv$")
#     f_fp = latest_match(FIGS_FP, r"^ClusterAssign_cumeig_gap_FP_.+\.csv$")
#     if f_oh is None: miss.append("figs_OH/ClusterAssign_cumeig_db_OH_*.csv")
#     if f_fp is None: miss.append("figs_FP/ClusterAssign_cumeig_gap_FP_*.csv")
#     if miss:
#         raise FileNotFoundError("必要ファイルが見つかりません:\n - " + "\n - ".join(miss))
#     return f_oh, f_fp

# # ========== 前処理：OH/FPの辞書 ==========
# def load_materials_features() -> pd.DataFrame:
#     """サンプル×材料 one-hot → material 列を付与（1つに決める）"""
#     X = read_csv_quiet(FEAT_OH)
#     if X is None: raise FileNotFoundError(FEAT_OH)
#     # サンプルid推定（左端 or 'Sample'）
#     if "Sample" in X.columns:
#         X = X.set_index("Sample")
#     else:
#         X = X.set_index(X.columns[0])
#     X.index = X.index.astype(str)  # ← 型を文字列に統一
#     # 1が立っている材料を抽出（複数1は最左の1を採用）
#     mat_cols = X.columns
#     mat_idx = X.apply(lambda r: next((c for c in mat_cols if r.get(c,0)==1), np.nan), axis=1)
#     df = pd.DataFrame({"sample": X.index.astype(str), "material": mat_idx.values}).set_index("sample")
#     return df

# def load_fragments_features() -> pd.DataFrame:
#     """サンプル×フラグメント one-hot"""
#     X = read_csv_quiet(FEAT_FP)
#     if X is None: raise FileNotFoundError(FEAT_FP)
#     if "Sample" in X.columns:
#         X = X.set_index("Sample")
#     else:
#         X = X.set_index(X.columns[0])
#     X.index = X.index.astype(str)  # ← 型を文字列に統一
#     return X

# def load_OH_material_cluster_map(file_oh: Path) -> pd.Series:
#     """材料 -> OHクラスタ"""
#     df = read_csv_quiet(file_oh)
#     if df is None: raise FileNotFoundError(file_oh)
#     # 列名吸収
#     cols = {c.lower(): c for c in df.columns}
#     vcol = cols.get("variable", df.columns[0])
#     ccol = cols.get("cluster", df.columns[-1])
#     s = pd.Series(df[ccol].values, index=df[vcol].astype(str).values, name="OH_cluster")
#     try: s = s.astype(int)
#     except: pass
#     return s

# def load_FP_fragment_cluster_map(file_fp: Path) -> pd.Series:
#     """フラグメント -> FPクラスタ"""
#     df = read_csv_quiet(file_fp)
#     if df is None: raise FileNotFoundError(file_fp)
#     cols = {c.lower(): c for c in df.columns}
#     vcol = cols.get("variable", df.columns[0])
#     ccol = cols.get("cluster",  df.columns[-1])
#     s = pd.Series(df[ccol].values, index=df[vcol].astype(str).values, name="FP_cluster")
#     try: s = s.astype(int)
#     except: pass
#     return s

# # ========== サンプルの FPクラスタ（多数決） ==========
# def sample_modal_fp_cluster(X_fp: pd.DataFrame, frag2cl: pd.Series) -> pd.Series:
#     """
#     各サンプルの「含有フラグメントのFPクラスタ」を多数決で1つに決定（同票は最小ID）。
#     X_fp: rows=samples, cols=fragments (0/1)
#     frag2cl: fragment -> FP_cluster
#     """
#     # 対象フラグメントのみに揃える
#     frag2cl = frag2cl.copy()
#     frag2cl.index = frag2cl.index.astype(str)
#     common_frags = [c for c in X_fp.columns if c in frag2cl.index]
#     if not common_frags:
#         return pd.Series(index=X_fp.index, dtype="float")
#     X = X_fp[common_frags]
#     cl_map = frag2cl[common_frags].astype("Int64")

#     out = []
#     for sid, row in X.iterrows():
#         active = row[row==1].index.tolist()
#         if not active:
#             out.append(np.nan); continue
#         labs = cl_map.loc[active].dropna().astype(int).values
#         if len(labs)==0:
#             out.append(np.nan); continue
#         vals, cnts = np.unique(labs, return_counts=True)
#         modal = int(vals[np.argmax(cnts)])
#         out.append(modal)
#     s = pd.Series(out, index=X.index, name="FP_cluster_modal")
#     s.index = s.index.astype(str)  # ← 型を文字列に統一
#     return s

# # ========== 指標（材料ごと） ==========
# def entropy(p):
#     p = np.asarray(p, dtype=float)
#     p = p[p>0]
#     if len(p)==0: return 0.0
#     return float(-(p*np.log2(p)).sum())

# def materials_fp_stats(sample2mat: pd.Series, sample2fp: pd.Series) -> pd.DataFrame:
#     """
#     各材料ごとに FPクラスタ分布から purity / entropy / n_samples を算出。
#     sample2mat: sample -> material
#     sample2fp : sample -> modal FP cluster (int or NaN)
#     """
#     # 追加: インデックス型を統一し共通サンプルに揃える
#     s_mat = sample2mat.copy(); s_fp = sample2fp.copy()
#     s_mat.index = s_mat.index.astype(str)
#     s_fp.index  = s_fp.index.astype(str)
#     common = s_mat.index.intersection(s_fp.index)
#     s_mat = s_mat.loc[common]
#     s_fp  = s_fp.loc[common]

#     df = pd.DataFrame({"material": s_mat, "FP": s_fp}).dropna()
#     rows=[]
#     for mat, sub in df.groupby("material"):
#         cnts = sub["FP"].value_counts().sort_values(ascending=False)
#         n = int(cnts.sum())
#         if n==0:
#             purity = np.nan; ent = np.nan; k = 0
#         else:
#             purity = float(cnts.iloc[0] / n)
#             p = (cnts / n).values
#             ent = entropy(p)
#             k = int(len(cnts))
#         rows.append({"material": mat, "FP_purity":purity, "FP_entropy":ent, "n_samples":n, "FP_modes":k})
#     return pd.DataFrame(rows)

# # ========== ペア指標（同一OHクラスタ内の材料ペア） ==========
# def pairwise_similarity_within_OH(material_stats: pd.DataFrame,
#                                   mat2OH: pd.Series,
#                                   sample2mat: pd.Series,
#                                   sample2fp: pd.Series) -> pd.DataFrame:
#     """
#     同一OHクラスタ内の材料ペアについて：
#       - FP_major_same（各材料のmodal FPが一致）
#       - cosine類似度（FPクラスタ分布ベクトル）
#       - JS距離（FPクラスタ分布ベクトル）
#     """
#     # サンプル側も型統一
#     s_mat = sample2mat.copy(); s_fp = sample2fp.copy()
#     s_mat.index = s_mat.index.astype(str)
#     s_fp.index  = s_fp.index.astype(str)
#     common = s_mat.index.intersection(s_fp.index)
#     s_mat = s_mat.loc[common]
#     s_fp  = s_fp.loc[common]

#     df_sf = pd.DataFrame({"material": s_mat, "FP": s_fp}).dropna()
#     if df_sf.empty:
#         return pd.DataFrame(columns=["OH_cluster","material_i","material_j","FP_major_same","cosine","JS"])

#     # クラスタ全集合
#     all_fp = sorted(df_sf["FP"].dropna().astype(int).unique().tolist())
#     # 材料別分布
#     vec = {}
#     for mat, sub in df_sf.groupby("material"):
#         counts = sub["FP"].astype(int).value_counts()
#         v = np.array([counts.get(c,0) for c in all_fp], dtype=float)
#         if v.sum()>0: v = v / v.sum()
#         vec[mat] = v

#     def cosine(a,b):
#         na = np.linalg.norm(a); nb = np.linalg.norm(b)
#         if na==0 or nb==0: return np.nan
#         return float(np.dot(a,b)/(na*nb))
#     def jsd(a,b):
#         a = np.asarray(a, float); b = np.asarray(b, float)
#         a = a / (a.sum()+1e-12); b = b / (b.sum()+1e-12)
#         m = 0.5*(a+b)
#         def KL(p,q):
#             mask = (p>0) & (q>0)
#             return np.sum(p[mask]*np.log2(p[mask]/q[mask]))
#         return float(0.5*KL(a,m)+0.5*KL(b,m))

#     # OHクラスタごとに材料ペア
#     mat2OH = mat2OH.dropna()
#     oh_groups = {}
#     for m, cl in mat2OH.items():
#         oh_groups.setdefault(int(cl), []).append(m)

#     rows=[]
#     for oh_c, mats in oh_groups.items():
#         mats = [m for m in mats if m in vec]
#         L = len(mats)
#         for i in range(L):
#             for j in range(i+1, L):
#                 mi, mj = mats[i], mats[j]
#                 vi, vj = vec.get(mi), vec.get(mj)
#                 if vi is None or vj is None: continue
#                 # modal FP
#                 modal_i = int(np.argmax(vi)) if vi.sum()>0 else -1
#                 modal_j = int(np.argmax(vj)) if vj.sum()>0 else -1
#                 major_same = int(modal_i==modal_j and modal_i!=-1)
#                 rows.append({
#                     "OH_cluster": oh_c,
#                     "material_i": mi,
#                     "material_j": mj,
#                     "FP_major_same": major_same,
#                     "cosine": cosine(vi,vj),
#                     "JS": jsd(vi,vj)
#                 })
#     return pd.DataFrame(rows)

# # ========== 可視化（重複ラベル回避＋色=OHクラスタ） ==========
# try:
#     from adjustText import adjust_text
#     _HAS_ADJUST = True
# except Exception:
#     _HAS_ADJUST = False

# def _select_top_for_labels(df, xcol, ycol, namecol, n=15):
#     tmp = df.copy()
#     r1 = (-tmp[ycol]).rank(method="first")  # purity高いほど小
#     r2 = ( tmp[xcol]).rank(method="first")  # entropy低いほど小
#     sc = r1 + r2
#     top = tmp.loc[sc.nsmallest(n).index].copy()
#     top["_x_lab"] = top[xcol].astype(float) + np.random.normal(scale=0.002, size=len(top))
#     top["_y_lab"] = top[ycol].astype(float) + np.random.normal(scale=0.002, size=len(top))
#     return top

# def plot_material_scatter_entropy_vs_purity(df_mat: pd.DataFrame,
#                                             mat2OH: pd.Series,
#                                             out_pdf: Path,
#                                             title="Materials — FP purity vs entropy (cumeig × gap)",
#                                             label_top_n: int = 15):
#     _set_jp_font()
#     # マージ
#     df = df_mat.merge(mat2OH.rename("OH_cluster").reset_index().rename(columns={"index":"material"}), on="material", how="left")
#     df = df.dropna(subset=["FP_purity","FP_entropy","OH_cluster"])
#     # 色 = OHクラスタ
#     cat = pd.Categorical(df["OH_cluster"].astype(int))
#     from matplotlib.cm import get_cmap
#     cmap = get_cmap("tab20")
#     colors = [cmap(i % 20) for i in range(max(len(cat.categories),1))]
#     color_map = {cat.categories[i]: colors[i % 20] for i in range(len(cat.categories))}
#     df["_color"] = [color_map.get(v, colors[0]) for v in cat]
#     # サイズ = n_samples（ルートスケール）
#     n = pd.to_numeric(df["n_samples"], errors="coerce").fillna(0.0)
#     sz = 36.0 * (np.sqrt(n)/(np.sqrt(n).quantile(0.9)+1e-9)).clip(0.3, 2.0)

#     fig, ax = plt.subplots(figsize=(8.2,6.0), dpi=300)
#     ax.scatter(df["FP_entropy"], df["FP_purity"], c=df["_color"], s=sz, alpha=0.85, edgecolors="none")
#     ax.set_xlabel("FP entropy (larger = more materials-spread)")
#     ax.set_ylabel("FP purity (larger = more materials-specific)")
#     ax.set_title(title)
#     ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)

#     # 凡例
#     import matplotlib.patches as mpatches
#     handles = [mpatches.Patch(color=color_map[v], label=f"OH {int(v)}") for v in cat.categories]
#     if handles:
#         ax.legend(handles=handles, title="Color = OH Cluster", loc="upper left",
#                   bbox_to_anchor=(1.02,1.0), borderaxespad=0.0)

#     # ラベル上位選抜
#     top = _select_top_for_labels(df, "FP_entropy", "FP_purity", "material", n=label_top_n)
#     texts=[]
#     for _, r in top.iterrows():
#         t = ax.text(r["_x_lab"], r["_y_lab"], f"{r['material']}", fontsize=8, ha="center", va="bottom")
#         texts.append(t)
#         if not _HAS_ADJUST:
#             ax.annotate("", xy=(r["_x_lab"], r["_y_lab"]-0.005), xytext=(r["_x_lab"], r["_y_lab"]-0.015),
#                         arrowprops=dict(arrowstyle="-", color="gray", lw=0.5))
#     if _HAS_ADJUST and texts:
#         adjust_text(texts,
#                     expand_points=(1.1,1.2),
#                     expand_text=(1.1,1.2),
#                     arrowprops=dict(arrowstyle="-", color="gray", lw=0.5),
#                     only_move={"points":"y","text":"xy"})

#     out_png = out_pdf.with_suffix(".png")
#     plt.tight_layout()
#     fig.savefig(out_png, bbox_inches="tight")
#     fig.savefig(out_pdf, bbox_inches="tight")
#     plt.close(fig)
#     print(f"[SAVE] {out_png}\n[SAVE] {out_pdf}")

# def plot_OH_cohesive_ratio_bar(df_pairs: pd.DataFrame, out_pdf: Path,
#                                title="OH clusters — cohesive ratio (cumeig × db)"):
#     _set_jp_font()
#     if df_pairs.empty:
#         print("[WARN] pair dataframe empty. Skip cohesion plot.")
#         return
#     coh = df_pairs.groupby("OH_cluster")["FP_major_same"].mean().reset_index().sort_values("OH_cluster")
#     fig, ax = plt.subplots(figsize=(8.0,4.8), dpi=300)
#     ax.bar(coh["OH_cluster"].astype(int), coh["FP_major_same"], width=0.8, alpha=0.9, color="#6c757d")
#     for x, y in zip(coh["OH_cluster"].astype(int), coh["FP_major_same"]):
#         ax.text(x, y+0.01, f"{y:.2f}", ha="center", va="bottom", fontsize=8)
#     ax.set_xlabel("OH cluster")
#     ax.set_ylabel("cohesive ratio (pair major-FP same rate)")
#     ax.set_ylim(0, 1.05)
#     ax.set_title(title)
#     ax.grid(axis="y", linestyle="--", linewidth=0.4, alpha=0.6)
#     out_png = out_pdf.with_suffix(".png")
#     plt.tight_layout()
#     fig.savefig(out_png, bbox_inches="tight")
#     fig.savefig(out_pdf, bbox_inches="tight")
#     plt.close(fig)
#     print(f"[SAVE] {out_png}\n[SAVE] {out_pdf}")

# def plot_pairwise_similarity_hist(df_pairs: pd.DataFrame, out_pdf: Path,
#                                   title="Within-OH pairs: FP_major_same / cosine / JS (cumeig × gap)"):
#     _set_jp_font()
#     if df_pairs.empty:
#         print("[WARN] pair dataframe empty. Skip hist plot.")
#         return
#     fig, axes = plt.subplots(1,3, figsize=(12,4.0), dpi=300)
#     # major same
#     mrate = df_pairs["FP_major_same"].mean() if "FP_major_same" in df_pairs else np.nan
#     axes[0].bar(["Different","Same"], [1-mrate if not np.isnan(mrate) else 0, mrate if not np.isnan(mrate) else 0],
#                 color=["#adb5bd","#4dabf7"], alpha=0.9)
#     axes[0].set_title(f"FP_major_same (rate={0.0 if np.isnan(mrate) else mrate:.2f})")
#     # cosine
#     axes[1].hist(df_pairs["cosine"].dropna(), bins=20, alpha=0.85, color="#198754")
#     axes[1].set_title("cosine similarity")
#     # JS
#     axes[2].hist(df_pairs["JS"].dropna(), bins=20, alpha=0.85, color="#dc3545")
#     axes[2].set_title("JS distance (lower=more similar)")
#     for ax in axes:
#         ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.6)
#         ax.set_ylabel("count")
#     for ax in axes[1:]:
#         ax.set_ylabel("")
#     fig.suptitle(title)
#     plt.tight_layout(rect=[0,0,1,0.93])
#     out_png = out_pdf.with_suffix(".png")
#     fig.savefig(out_png, bbox_inches="tight")
#     fig.savefig(out_pdf, bbox_inches="tight")
#     plt.close(fig)
#     print(f"[SAVE] {out_png}\n[SAVE] {out_pdf}")

# # ========== 要約テーブル ==========
# def save_summary_table(material_stats: pd.DataFrame,
#                        df_pairs_gap: pd.DataFrame,
#                        df_pairs_db: pd.DataFrame,
#                        out_csv: Path):
#     """
#     本文表用に主要指標を集約：
#       - mean_FP_purity（材料→構造の凝集）
#       - pair_FP_major_same_rate（gapでの平均）
#       - OHcluster_median_cohesive_ratio（dbでのmedian）
#     """
#     mean_purity = material_stats["FP_purity"].mean()
#     rate_gap = df_pairs_gap["FP_major_same"].mean() if not df_pairs_gap.empty else np.nan
#     if not df_pairs_db.empty:
#         coh_db = df_pairs_db.groupby("OH_cluster")["FP_major_same"].mean()
#         med_coh = coh_db.median()
#     else:
#         med_coh = np.nan

#     pd.DataFrame([{
#         "mode":"cumeig", "index_gap":"gap", "index_db":"db",
#         "mean_FP_purity": round(float(mean_purity), 3) if not pd.isna(mean_purity) else np.nan,
#         "pair_FP_major_same_rate": round(float(rate_gap), 3) if not pd.isna(rate_gap) else np.nan,
#         "OHcluster_median_cohesive_ratio": round(float(med_coh), 3) if not pd.isna(med_coh) else np.nan
#     }]).to_csv(out_csv, index=False, encoding="utf-8-sig")
#     print(f"[SAVE] {out_csv}")

# # ========== MAIN ==========
# def main():
#     _set_jp_font()
#     # 入力チェック & ロード
#     f_oh, f_fp = ensure_inputs()
#     mat_map = load_OH_material_cluster_map(f_oh)             # material -> OH_cluster（cumeig×db）
#     frag_map = load_FP_fragment_cluster_map(f_fp)            # fragment -> FP_cluster（cumeig×gap）
#     df_mat = load_materials_features()                       # sample -> material（indexはstr）
#     X_fp   = load_fragments_features()                       # sample x fragment (0/1)（indexはstr）

#     # サンプルの modal FP クラスタ
#     s_modal_fp = sample_modal_fp_cluster(X_fp, frag_map)     # sample -> FP_modal_cluster
#     # 念のため index を str に統一（保険）
#     s_modal_fp.index = s_modal_fp.index.astype(str)
#     df_mat.index = df_mat.index.astype(str)

#     # 材料ごとの purity / entropy
#     mat_stats = materials_fp_stats(df_mat["material"], s_modal_fp)

#     # Fig(b): purity–entropy（色=OHクラスタ、ラベル重なり回避）
#     out_b = OUT_GAP / "Fig_material_scatter_entropy_vs_purity.pdf"
#     plot_material_scatter_entropy_vs_purity(mat_stats, mat_map, out_pdf=out_b,
#                                             title="Materials — FP purity vs entropy (cumeig × gap)",
#                                             label_top_n=15)

#     # ペア指標：gap（FP_major_same・cosine・JS）
#     pairs_gap = pairwise_similarity_within_OH(mat_stats, mat_map, df_mat["material"], s_modal_fp)
#     out_d = OUT_GAP / "Fig_pairwise_similarity_hist.pdf"
#     plot_pairwise_similarity_hist(pairs_gap, out_pdf=out_d,
#                                   title="Within-OH pairs: FP_major_same / cosine / JS (cumeig × gap)")

#     # cohesive ratio：db（OHクラスタごとの modal FP一致率）→ pairs_gap を使って集計
#     pairs_db = pairs_gap.copy()
#     out_c = OUT_DB / "Fig_OHcluster_cohesive_ratio.pdf"
#     plot_OH_cohesive_ratio_bar(pairs_db, out_pdf=out_c,
#                                title="OH clusters — cohesive ratio (cumeig × db)")

#     # 本文テーブル（要約）
#     out_tab = OUT_BASE / "Table_4.2.4.4_summary_all_conditions.csv"
#     save_summary_table(mat_stats, pairs_gap, pairs_db, out_csv=out_tab)

#     print("🎉 Done: 4.2.4.4(b)(c)(d) + summary table exported under OHFP_varmap/")

# if __name__ == "__main__":
#     # Jupyterでも動くように argv を抑制
#     import sys
#     sys.argv = ["paper_4.2.4.4_build_onlyMat_main.py"]
#     main()

/tmp/ipykernel_3936950/3005393958.py:319: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = get_cmap("tab20")


[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/figs_cumeig_gap/Fig_material_scatter_entropy_vs_purity.png
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/figs_cumeig_gap/Fig_material_scatter_entropy_vs_purity.pdf
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/figs_cumeig_gap/Fig_pairwise_similarity_hist.png
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/figs_cumeig_gap/Fig_pairwise_similarity_hist.pdf
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/figs_cumeig_db/Fig_OHcluster_cohesive_ratio.png
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/figs_cumeig_db/Fig_OHcluster_cohesive_ratio.pdf
[SAVE] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/OHFP_varmap/Table_4.2.4.4_summary_all_conditions.csv
🎉 Done: 4.2.4.4(b)(c)(d) + summary table exported under OHFP_varmap/
